In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Ventes").getOrCreate()

data = """date,region,produit,quantite,prix
2025-10-19,Guelmim-Oued Noun,Laptop,10,10401
2025-10-26,Tanger-Tétouan-Al Hoceïma,Imprimante,2,2230
2025-10-15,Fès-Meknès,Casque,5,775
2025-10-02,Fès-Meknès,Imprimante,9,3129
2025-10-02,Marrakech-Safi,Laptop,1,8514
2025-10-30,Tanger-Tétouan-Al Hoceïma,USB,9,85
2025-10-19,Fès-Meknès,Laptop,4,9466
2025-10-25,Marrakech-Safi,Imprimante,9,2216
"""
with open("ventes.csv", "w") as f:
    f.write(data)

df = spark.read.option("header", True).option("inferSchema", True).csv("ventes.csv")
df.show()
df.printSchema()

+----------+--------------------+----------+--------+-----+
|      date|              region|   produit|quantite| prix|
+----------+--------------------+----------+--------+-----+
|2025-10-19|   Guelmim-Oued Noun|    Laptop|      10|10401|
|2025-10-26|Tanger-Tétouan-Al...|Imprimante|       2| 2230|
|2025-10-15|          Fès-Meknès|    Casque|       5|  775|
|2025-10-02|          Fès-Meknès|Imprimante|       9| 3129|
|2025-10-02|      Marrakech-Safi|    Laptop|       1| 8514|
|2025-10-30|Tanger-Tétouan-Al...|       USB|       9|   85|
|2025-10-19|          Fès-Meknès|    Laptop|       4| 9466|
|2025-10-25|      Marrakech-Safi|Imprimante|       9| 2216|
+----------+--------------------+----------+--------+-----+

root
 |-- date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- produit: string (nullable = true)
 |-- quantite: integer (nullable = true)
 |-- prix: integer (nullable = true)



**1: Vue temporaire ventes_view**

In [2]:
df.createOrReplaceTempView("ventes_view")
spark.sql("SELECT * FROM ventes_view").show()

+----------+--------------------+----------+--------+-----+
|      date|              region|   produit|quantite| prix|
+----------+--------------------+----------+--------+-----+
|2025-10-19|   Guelmim-Oued Noun|    Laptop|      10|10401|
|2025-10-26|Tanger-Tétouan-Al...|Imprimante|       2| 2230|
|2025-10-15|          Fès-Meknès|    Casque|       5|  775|
|2025-10-02|          Fès-Meknès|Imprimante|       9| 3129|
|2025-10-02|      Marrakech-Safi|    Laptop|       1| 8514|
|2025-10-30|Tanger-Tétouan-Al...|       USB|       9|   85|
|2025-10-19|          Fès-Meknès|    Laptop|       4| 9466|
|2025-10-25|      Marrakech-Safi|Imprimante|       9| 2216|
+----------+--------------------+----------+--------+-----+



**Question 2: Unités vendues par région**

In [3]:
spark.sql("""
SELECT region, SUM(quantite) AS total_unites
FROM ventes_view
GROUP BY region
ORDER BY total_unites DESC
""").show()

+--------------------+------------+
|              region|total_unites|
+--------------------+------------+
|          Fès-Meknès|          18|
|Tanger-Tétouan-Al...|          11|
|   Guelmim-Oued Noun|          10|
|      Marrakech-Safi|          10|
+--------------------+------------+



**Question 3: Prix max par produit + produit le plus cher**

In [4]:
# Prix maximum pour chaque produit
spark.sql("""
SELECT produit, MAX(prix) AS prix_max
FROM ventes_view
GROUP BY produit
ORDER BY prix_max DESC
""").show()

# Le produit le plus cher du dataset
spark.sql("""
SELECT produit, prix AS prix_max
FROM ventes_view
ORDER BY prix DESC
LIMIT 1
""").show()

+----------+--------+
|   produit|prix_max|
+----------+--------+
|    Laptop|   10401|
|Imprimante|    3129|
|    Casque|     775|
|       USB|      85|
+----------+--------+

+-------+--------+
|produit|prix_max|
+-------+--------+
| Laptop|   10401|
+-------+--------+



**Question 4: Colonne calculée total_vente + CA par produit**

In [5]:
spark.sql("""
SELECT produit, SUM(total_vente) AS chiffre_affaires
FROM (
    SELECT *, quantite * prix AS total_vente
    FROM ventes_view
)
GROUP BY produit
ORDER BY chiffre_affaires DESC
""").show()

+----------+----------------+
|   produit|chiffre_affaires|
+----------+----------------+
|    Laptop|          150388|
|Imprimante|           52565|
|    Casque|            3875|
|       USB|             765|
+----------+----------------+



**Question 5: Analyse mensuelle (année 2025)**

In [6]:
# Extraire l'année et le mois
spark.sql("""
SELECT date,
       SUBSTRING(date, 1, 4) AS annee,
       SUBSTRING(date, 6, 2) AS mois
FROM ventes_view
""").show()

# CA total par mois pour 2025
spark.sql("""
SELECT SUBSTRING(date, 1, 4) AS annee,
       SUBSTRING(date, 6, 2) AS mois,
       SUM(quantite * prix) AS chiffre_affaires
FROM ventes_view
WHERE SUBSTRING(date, 1, 4) = '2025'
GROUP BY SUBSTRING(date, 1, 4), SUBSTRING(date, 6, 2)
ORDER BY mois
""").show()

+----------+-----+----+
|      date|annee|mois|
+----------+-----+----+
|2025-10-19| 2025|  10|
|2025-10-26| 2025|  10|
|2025-10-15| 2025|  10|
|2025-10-02| 2025|  10|
|2025-10-02| 2025|  10|
|2025-10-30| 2025|  10|
|2025-10-19| 2025|  10|
|2025-10-25| 2025|  10|
+----------+-----+----+

+-----+----+----------------+
|annee|mois|chiffre_affaires|
+-----+----+----------------+
| 2025|  10|          207593|
+-----+----+----------------+



**Question 6: Ventes où total_vente > 10000**

In [7]:
spark.sql("""
SELECT *
FROM (
    SELECT *, quantite * prix AS total_vente
    FROM ventes_view
)
WHERE total_vente > 10000
""").show()

+----------+-----------------+----------+--------+-----+-----------+
|      date|           region|   produit|quantite| prix|total_vente|
+----------+-----------------+----------+--------+-----+-----------+
|2025-10-19|Guelmim-Oued Noun|    Laptop|      10|10401|     104010|
|2025-10-02|       Fès-Meknès|Imprimante|       9| 3129|      28161|
|2025-10-19|       Fès-Meknès|    Laptop|       4| 9466|      37864|
|2025-10-25|   Marrakech-Safi|Imprimante|       9| 2216|      19944|
+----------+-----------------+----------+--------+-----+-----------+



**Question 7: Produit le plus vendu (quantité) par région**

In [8]:
spark.sql("""
WITH qte_par_produit AS (
    SELECT region, produit, SUM(quantite) AS qte_totale
    FROM ventes_view
    GROUP BY region, produit
),
classement AS (
    SELECT region, produit, qte_totale,
           RANK() OVER (PARTITION BY region ORDER BY qte_totale DESC) AS rang
    FROM qte_par_produit
)
SELECT region, produit, qte_totale
FROM classement
WHERE rang = 1
""").show()

+--------------------+----------+----------+
|              region|   produit|qte_totale|
+--------------------+----------+----------+
|          Fès-Meknès|Imprimante|         9|
|   Guelmim-Oued Noun|    Laptop|        10|
|      Marrakech-Safi|Imprimante|         9|
|Tanger-Tétouan-Al...|       USB|         9|
+--------------------+----------+----------+



**Question 8: Min, Max, Moyenne des prix par produit**

In [9]:
spark.sql("""
SELECT produit,
       MIN(prix) AS prix_min,
       MAX(prix) AS prix_max,
       ROUND(AVG(prix), 2) AS prix_moyen
FROM ventes_view
GROUP BY produit
""").show()

+----------+--------+--------+----------+
|   produit|prix_min|prix_max|prix_moyen|
+----------+--------+--------+----------+
|    Laptop|    8514|   10401|   9460.33|
|Imprimante|    2216|    3129|    2525.0|
|    Casque|     775|     775|     775.0|
|       USB|      85|      85|      85.0|
+----------+--------+--------+----------+



**Question 9: Vue ventes_analyse**

In [10]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW ventes_analyse AS
SELECT region, produit, quantite, prix,
       quantite * prix AS total_vente,
       SUBSTRING(date, 1, 4) AS annee,
       SUBSTRING(date, 6, 2) AS mois
FROM ventes_view
""")

spark.sql("SELECT * FROM ventes_analyse").show()

+--------------------+----------+--------+-----+-----------+-----+----+
|              region|   produit|quantite| prix|total_vente|annee|mois|
+--------------------+----------+--------+-----+-----------+-----+----+
|   Guelmim-Oued Noun|    Laptop|      10|10401|     104010| 2025|  10|
|Tanger-Tétouan-Al...|Imprimante|       2| 2230|       4460| 2025|  10|
|          Fès-Meknès|    Casque|       5|  775|       3875| 2025|  10|
|          Fès-Meknès|Imprimante|       9| 3129|      28161| 2025|  10|
|      Marrakech-Safi|    Laptop|       1| 8514|       8514| 2025|  10|
|Tanger-Tétouan-Al...|       USB|       9|   85|        765| 2025|  10|
|          Fès-Meknès|    Laptop|       4| 9466|      37864| 2025|  10|
|      Marrakech-Safi|Imprimante|       9| 2216|      19944| 2025|  10|
+--------------------+----------+--------+-----+-----------+-----+----+



**Question 10: Top 3 produits par chiffre d’affaires**

In [11]:
spark.sql("""
SELECT produit, SUM(total_vente) AS chiffre_affaires
FROM ventes_analyse
GROUP BY produit
ORDER BY chiffre_affaires DESC
LIMIT 3
""").show()

+----------+----------------+
|   produit|chiffre_affaires|
+----------+----------------+
|    Laptop|          150388|
|Imprimante|           52565|
|    Casque|            3875|
+----------+----------------+



**Question 11: CA par région + meilleure région**

In [12]:
# CA total par région
spark.sql("""
SELECT region, SUM(total_vente) AS chiffre_affaires
FROM ventes_analyse
GROUP BY region
ORDER BY chiffre_affaires DESC
""").show()

# Région avec le plus haut CA
spark.sql("""
SELECT region, SUM(total_vente) AS chiffre_affaires
FROM ventes_analyse
GROUP BY region
ORDER BY chiffre_affaires DESC
LIMIT 1
""").show()

+--------------------+----------------+
|              region|chiffre_affaires|
+--------------------+----------------+
|   Guelmim-Oued Noun|          104010|
|          Fès-Meknès|           69900|
|      Marrakech-Safi|           28458|
|Tanger-Tétouan-Al...|            5225|
+--------------------+----------------+

+-----------------+----------------+
|           region|chiffre_affaires|
+-----------------+----------------+
|Guelmim-Oued Noun|          104010|
+-----------------+----------------+

